In [ ]:
from pathlib import Path
import json

# ==========================================
# 1. EXPORT DIRECTORY
# ==========================================

EXPORT_DIR = (
    PROJECT_DIR
    / "models"
    / "karakalpak_xlsr300m_lora_ctc_v1"
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ==========================================
# 2. SAVE LORA + CTC HEAD
# ==========================================

full_model.save_pretrained(
    str(EXPORT_DIR),
    safe_serialization=True,
    save_embedding_layers=False
)

# ==========================================
# 3. SAVE PROCESSOR
# ==========================================

processor.save_pretrained(
    str(EXPORT_DIR / "processor")
)

# ==========================================
# 4. SAVE METADATA
# ==========================================

metadata = {
    "base_model": "facebook/wav2vec2-xls-r-300m",
    "checkpoint_step": 13023,
    "vocab_size": 36,
    "lora_rank": 8,
    "lora_alpha": 16,
    "validation_wer_greedy": 50.0032,
    "validation_cer_greedy": 11.5323,
    "validation_wer_kenlm": 27.0632,
    "validation_cer_kenlm": 10.0256,
}

with open(
    EXPORT_DIR / "model_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=4,
        ensure_ascii=False
    )

# ==========================================
# 5. VERIFY EXPORTED FILES
# ==========================================

print("=" * 60)
print("KARAKALPAK STT MODEL EXPORT")
print("=" * 60)

for path in sorted(EXPORT_DIR.rglob("*")):
    if path.is_file():
        print(
            "✅",
            path.relative_to(EXPORT_DIR)
        )

assert (
    EXPORT_DIR / "adapter_model.safetensors"
).exists()

assert (
    EXPORT_DIR / "adapter_config.json"
).exists()

print("\n✅ Model exported successfully!")
print("Location:", EXPORT_DIR)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")

print("✅ Project:", PROJECT_DIR)
print("✅ Exists:", PROJECT_DIR.exists())

print("full_model:", "full_model" in globals())
print("processor:", "processor" in globals())

In [ ]:
import torch

from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from peft import LoraConfig, get_peft_model

# 1. Checkpoint
checkpoint_path = (
    PROJECT_DIR
    / "checkpoints"
    / "full_lora_step_13023"
    / "training_state.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

# 2. Processor
processor = Wav2Vec2Processor.from_pretrained(
    str(PROJECT_DIR / "processor")
)

# 3. Base XLS-R model
base_model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    vocab_size=len(processor.tokenizer),
    pad_token_id=processor.tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True
)

base_model.config.layerdrop = 0.0

for param in base_model.parameters():
    param.requires_grad = False

# 4. LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    modules_to_save=["lm_head"]
)

full_model = get_peft_model(
    base_model,
    lora_config
)

# 5. Checkpoint weightlarini tiklash
model_parameters = dict(full_model.named_parameters())
saved_weights = checkpoint["model_weights"]

assert set(saved_weights).issubset(model_parameters), (
    "Checkpoint parameter names do not match!"
)

with torch.no_grad():
    for name, tensor in saved_weights.items():
        parameter = model_parameters[name]

        assert parameter.shape == tensor.shape, (
            f"Shape mismatch: {name}"
        )

        parameter.copy_(
            tensor.to(
                device=parameter.device,
                dtype=parameter.dtype
            )
        )

# 6. GPU va inference
device = "cuda" if torch.cuda.is_available() else "cpu"

full_model = full_model.to(device)
full_model.eval()

print("=" * 55)
print("KARAKALPAK XLS-R-300M RESTORED")
print("=" * 55)
print("Step:", checkpoint["global_step"])
print("Loaded tensors:", len(saved_weights))
print("Vocabulary:", len(processor.tokenizer))
print("Device:", device)

assert checkpoint["global_step"] == 13023
assert len(saved_weights) == 98

print("\n✅ Model successfully restored!")

In [ ]:
EXPORT_DIR = (
    PROJECT_DIR
    / "models"
    / "karakalpak_xlsr300m_lora_ctc_v1"
)

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

full_model.save_pretrained(
    str(EXPORT_DIR),
    safe_serialization=True,
    save_embedding_layers=False
)

processor.save_pretrained(
    str(EXPORT_DIR / "processor")
)

print("✅ Model exported:", EXPORT_DIR)

In [ ]:
from safetensors import safe_open

adapter_path = EXPORT_DIR / "adapter_model.safetensors"

assert adapter_path.exists(), "Adapter fayli topilmadi!"

with safe_open(str(adapter_path), framework="pt", device="cpu") as f:
    keys = list(f.keys())

    lora_keys = [k for k in keys if "lora_" in k]
    head_keys = [k for k in keys if "lm_head" in k]

    lora_params = sum(
        f.get_tensor(k).numel() for k in lora_keys
    )

    head_params = sum(
        f.get_tensor(k).numel() for k in head_keys
    )

print("=" * 55)
print("EXPORTED MODEL VERIFICATION")
print("=" * 55)

print("Total tensors:", len(keys))
print("LoRA tensors:", len(lora_keys))
print("CTC Head tensors:", len(head_keys))

print("\nLoRA parameters:", f"{lora_params:,}")
print("CTC Head parameters:", f"{head_params:,}")

print("\nCTC Head keys:")
for key in head_keys:
    print(key)

assert lora_params == 786432, "LoRA parametrlari to‘liq emas!"
assert head_params == 36900, "CTC Head parametrlari to‘liq emas!"

print("\n✅ LoRA + CTC Head successfully verified!")

In [ ]:
import json
import importlib
import sys

from pathlib import Path
from pyctcdecode import build_ctcdecoder

# 1. KenLM fayllari
KENLM_DIR = PROJECT_DIR / "kenlm"

lm_path = KENLM_DIR / "karakalpak_5gram.binary"
config_path = KENLM_DIR / "decoder_config.json"

assert lm_path.exists(), "KenLM binary topilmadi!"
assert config_path.exists(), "Decoder config topilmadi!"

# 2. Saqlangan decoder parametrlari
with open(config_path, encoding="utf-8") as f:
    decoder_config = json.load(f)

print("Decoder configuration:")
print(json.dumps(decoder_config, indent=2, ensure_ascii=False))

# 3. Tokenlarni ID tartibida olamiz
vocab = processor.tokenizer.get_vocab()

labels = [
    token
    for token, token_id in sorted(
        vocab.items(),
        key=lambda item: item[1]
    )
]

assert len(labels) == 36

# 4. KenLM decoder
kenlm_decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=str(lm_path),
    alpha=0.5,
    beta=1.0
)

print("\n✅ KenLM decoder restored!")
print("Vocabulary:", len(labels))
print("KenLM model:", lm_path.name)
print("Alpha:", 0.5)
print("Beta:", 1.0)

In [ ]:
from pyctcdecode import build_ctcdecoder

# ==========================================
# 1. KENLM TRAINING CORPUS
# ==========================================

corpus_path = (
    PROJECT_DIR
    / "kenlm"
    / "karakalpak_train.txt"
)

assert corpus_path.exists()

# ==========================================
# 2. BUILD UNIGRAM VOCABULARY
# ==========================================

unigrams = set()

with open(corpus_path, encoding="utf-8") as f:
    for line in f:
        unigrams.update(line.strip().split())

print("Unique unigrams:", len(unigrams))

# ==========================================
# 3. RESTORE DECODER WITH UNIGRAMS
# ==========================================

kenlm_decoder = build_ctcdecoder(
    labels=labels,
    kenlm_model_path=str(lm_path),
    unigrams=sorted(unigrams),
    alpha=0.5,
    beta=1.0
)

print("\n✅ KenLM decoder rebuilt with unigrams!")
print("Unigrams:", len(unigrams))
print("Beam width baseline:", 10)

In [ ]:
import sys
import numpy as np
from importlib.metadata import version

print("Python:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("NumPy:", np.__version__)
print("PyArrow:", version("pyarrow"))
print("Pandas:", version("pandas"))
print("PyTorch:", version("torch"))
print("Transformers:", version("transformers"))

In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch

from pathlib import Path
from importlib.metadata import version

PROJECT_DIR = Path("/home/intern/projects/karakalpak_stt")

print("=" * 55)
print("ENVIRONMENT CHECK")
print("=" * 55)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pa.__version__)
print("PyTorch:", torch.__version__)
print("pyctcdecode:", version("pyctcdecode"))
print("datasets:", version("datasets"))

# Validation metadata
val_path = PROJECT_DIR / "data" / "val_metadata.parquet"

val_df = pd.read_parquet(val_path)

print("\nValidation samples:", len(val_df))
print("Validation columns:", val_df.columns.tolist())

# Audio shard
shard_files = sorted(
    (PROJECT_DIR / "audio_shards").glob("*.parquet")
)

print("\nAudio shards:", len(shard_files))

if shard_files:
    schema = pq.ParquetFile(shard_files[0]).schema_arrow

    print("\nAudio shard schema:")
    print(schema)

print("\n✅ Validation dataset successfully loaded!")